# Quality Assessment of Book Descriptions

## Heuristic Filtering

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/goodreads_library_descriptions_cleaned.csv")

In [3]:
# Set pandas display options
pd.set_option('display.max_colwidth', None)  # show full text
pd.set_option('display.width', 2000)        # wide display so no wrapping

In [65]:
!pip install sentence-transformers faiss-cpu umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 12.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 12.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 13.0 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [umap-learn]5 [umap-learn]


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

batch_size = 64
embeddings = []

descriptions = df['Description'].fillna("").tolist()

for i in range(0, len(descriptions), batch_size):
    batch = descriptions[i:i+batch_size]
    batch_emb = model.encode(batch, show_progress_bar=False, normalize_embeddings=True)
    embeddings.append(batch_emb)

embeddings = np.vstack(embeddings)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a52036f9-4de6-4c4b-b998-782228109821)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./config_sentence_transformers.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 3b476c99-5d76-4599-9776-b6625df0bd98)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./config_sentence_transformers.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: b971b06a-8cd9-46db-b212-efd0e40b8209)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main

In [5]:
import faiss

d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)  # cosine similarity; CPU by default
index.add(embeddings.astype('float32'))

k = 10  # neighbors
similarities, indices = index.search(embeddings, k)


: 